In [4]:
!pip install --upgrade pip setuptools
!pip install -U openmim
!mim install mmengine
!mim install "mmcv>=2.0.0"

!git clone https://github.com/open-mmlab/mmdetection.git
%cd mmdetection
!pip install -e .

!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 50.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 60.2.0
    Uninstalling setuptools-60.2.0:
      Successfully uninstalled setuptools-60.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
openxlab 0.1.3 requires rich~=13.4.2, but you have rich 15.0.0 which is incompatible.
openxlab 0.1.3 requires setuptools~=60.2.0, but you have setuptools 84.0.0 which is incompatible.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
torch 2.11.0+cpu

  Using cached rich-13.4.2-py3-none-any.whl.metadata (18 kB)
  Using cached setuptools-60.2.0-py3-none-any.whl.metadata (5.1 kB)
Using cached rich-13.4.2-py3-none-any.whl (239 kB)
Using cached setuptools-60.2.0-py3-none-any.whl (953 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0 0/2 [setuptools]
  Attempting uninstall: rich━━━━━━━━━━━━━━━━━━━━ 0/2 [setuptools]
    Found existing installation: rich 15.0.0 0/2 [setuptools]
    Uninstalling rich-15.0.0:━━━━━━━━━━━━━━━ 0/2 [setuptools]
      Successfully uninstalled rich-15.0.0━━ 0/2 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rich]1/2 [rich]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
typer 0.25.1 requires r

Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.13/dist-packages/mim/__init__.py", line 10, in <module>
    import setuptools  # noqa: F401
    ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/setuptools/__init__.py", line 16, in <module>
    import setuptools.version
  File "/usr/local/lib/python3.13/dist-packages/setuptools/version.py", line 1, in <module>
    import pkg_resources
  File "/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py", line 2172, in <module>
    register_finder(pkgutil.ImpImporter, find_on_path)
                    ^^^^^^^^^^^^^^^^^^^
AttributeError: module 'pkgutil' has no attribute 'ImpImporter'. Did you mean: 'zipimporter'?
Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.13/dist-packages/mim/__init__.py", line 10, in <module>
    import setu

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API = userdata.get('ROBOFLOW_API')
rf = Roboflow(api_key=ROBOFLOW_API)
project = rf.workspace("udy").project("visdrone-qr0vn")
version = project.version(2)
dataset = version.download("coco-mmdetection")

In [5]:
!ls

CITATION.cff	   docs		    projects	     requirements.txt  tools
configs		   LICENSE	    pytest.ini	     resources
dataset-index.yml  MANIFEST.in	    README.md	     setup.cfg
demo		   mmdet	    README_zh-CN.md  setup.py
docker		   model-index.yml  requirements     tests


In [ ]:
%%writefile configs/cascade_rcnn/cascade-rcnn_r50_fpn_1x_roboflow.py
_base_ = [
    '../_base_/models/cascade-rcnn_r50_fpn.py',
    '../_base_/datasets/coco_detection.py',
    '../_base_/schedules/schedule_1x.py', 
    '../_base_/default_runtime.py'
]

# Định nghĩa 10 classes chuẩn của VisDrone
classes = ('pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor')

# Cấu hình lại mô hình với 10 classes
model = dict(
    roi_head=dict(
        bbox_head=[
            dict(
                type='Shared2FCBBoxHead',
                in_channels=256,
                fc_out_channels=1024,
                roi_feat_size=7,
                num_classes=10, 
                bbox_coder=dict(type='DeltaXYWHBBoxCoder', target_means=[0., 0., 0., 0.], target_stds=[0.1, 0.1, 0.2, 0.2]),
                reg_class_agnostic=True,
                loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
                loss_bbox=dict(type='SmoothL1Loss', beta=1.0, loss_weight=1.0)),
            dict(
                type='Shared2FCBBoxHead',
                in_channels=256,
                fc_out_channels=1024,
                roi_feat_size=7,
                num_classes=10,
                bbox_coder=dict(type='DeltaXYWHBBoxCoder', target_means=[0., 0., 0., 0.], target_stds=[0.05, 0.05, 0.1, 0.1]),
                reg_class_agnostic=True,
                loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
                loss_bbox=dict(type='SmoothL1Loss', beta=1.0, loss_weight=1.0)),
            dict(
                type='Shared2FCBBoxHead',
                in_channels=256,
                fc_out_channels=1024,
                roi_feat_size=7,
                num_classes=10,
                bbox_coder=dict(type='DeltaXYWHBBoxCoder', target_means=[0., 0., 0., 0.], target_stds=[0.033, 0.033, 0.067, 0.067]),
                reg_class_agnostic=True,
                loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
                loss_bbox=dict(type='SmoothL1Loss', beta=1.0, loss_weight=1.0))
        ]))

# --- CẤU HÌNH DATASET ROBOFLOW ---
# Thay đổi thư mục này thành tên thư mục chứa data thật sự của bạn sau khi tải bằng Roboflow 
# (Thường là tên của version, ví dụ: 'VisDrone-2/')
data_root = 'Visdrone-2/' 
metainfo = dict(classes=classes)

train_dataloader = dict(
    batch_size=2, 
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='train/_annotations.coco.json',
        data_prefix=dict(img='train/')))

val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='valid/_annotations.coco.json', 
        data_prefix=dict(img='valid/')))

test_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='test/_annotations.coco.json',
        data_prefix=dict(img='test/')))

val_evaluator = dict(ann_file=data_root + 'valid/_annotations.coco.json')
test_evaluator = dict(ann_file=data_root + 'test/_annotations.coco.json')

load_from = 'https://download.openmmlab.com/mmdetection/v2.0/cascade_rcnn/cascade_rcnn_r50_fpn_1x_coco/cascade_rcnn_r50_fpn_1x_coco_20200316-3dc56deb.pth'

train_cfg = dict(max_epochs=12, val_interval=1)
